# 72 - Soft Label Training on Late Fusion TL (Target: beat 0.567)

**Motivasi:** nb 71 menunjukkan KL-divergence soft label di CNN TL 4c B1 boost Macro F1 **+0.09** (0.427 → 0.517). Extend ke arsitektur best overall — Late Fusion TL — dengan target **beat 0.567** (Late Fusion TL 4c B3 hard CE existing best).

**Strategi:**
1. Train **CNN TL** + **FCNN** separately dengan soft label loss (2 variants: KL-div & Soft CE)
2. Inference di Primer val + test, softmax averaging
3. Grid search `w` di Primer val, eval di Primer test
4. Compare vs existing Late Fusion TL hard baselines

**Eksperimen (4-class):**

| Config | CNN_TL Loss | FCNN Loss | Note |
|--------|-------------|-----------|------|
| A_KL_div | KL-div | KL-div | winning loss dari nb 71 |
| B_soft_CE | Soft CE | Soft CE | runner-up di nb 71 |

**Baseline references:**
- Late Fusion TL 4c B1 hard CE (existing): **0.513**
- Late Fusion TL 4c B3 hard CE (current best overall): **0.567** ⭐ target
- CNN TL 4c B1 soft KL (nb 71): 0.517

**Hipotesis:** Soft label + decision-level fusion → independen learning di setiap branch, soft target di keduanya meningkatkan diversitas + better calibration → beat 0.567.

**Output:** `models/frontonly_conf60/soft_label/soft_lf_tl_4c_results.json`

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionCNNTransfer, EmotionFCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DATA_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'frontonly_conf60' / 'soft_label'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
LR_TL = 0.00005
LR_FCNN = 0.0001

EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']
REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)
NUM_CLASSES = 4

print('Setup complete.')

Device: cuda
Setup complete.


In [2]:
# ── Load data (4-class with soft labels) ──

def remap_soft_to_4class(y_soft_7):
    n = len(y_soft_7)
    y_soft_4 = np.zeros((n, 4), dtype=np.float32)
    y_soft_4[:, 0] = y_soft_7[:, 0]
    y_soft_4[:, 1] = y_soft_7[:, 1]
    y_soft_4[:, 2] = y_soft_7[:, 2]
    y_soft_4[:, 3] = y_soft_7[:, 3:7].sum(axis=1)
    return y_soft_4


def load_split(split):
    img = np.load(DATA_DIR / f'X_{split}_images.npy')
    lm  = np.load(DATA_DIR / f'X_{split}_landmarks.npy')
    y7  = np.load(DATA_DIR / f'y_{split}.npy')
    ys7 = np.load(DATA_DIR / f'y_{split}_soft.npy')
    return img, lm, REMAP_4[y7], remap_soft_to_4class(ys7)


X_tr_img, X_tr_lm, y_tr, y_tr_soft = load_split('train')
X_v_img,  X_v_lm,  y_v,  y_v_soft  = load_split('val')
X_te_img, X_te_lm, y_te, y_te_soft = load_split('test')

print(f'Train: img={X_tr_img.shape}  lm={X_tr_lm.shape}  y_hard={y_tr.shape}  y_soft={y_tr_soft.shape}')
print(f'Val:   img={X_v_img.shape}   lm={X_v_lm.shape}    y_hard={y_v.shape}')
print(f'Test:  img={X_te_img.shape}  lm={X_te_lm.shape}   y_hard={y_te.shape}')
print(f'\n4-class hard dist (train): {np.bincount(y_tr, minlength=4).tolist()}')
print(f'Soft label sum per sample: mean={y_tr_soft.sum(axis=1).mean():.4f}  min={y_tr_soft.sum(axis=1).min():.4f}')

Train: img=(5287, 224, 224, 3)  lm=(5287, 136)  y_hard=(5287,)  y_soft=(5287, 4)
Val:   img=(579, 224, 224, 3)   lm=(579, 136)    y_hard=(579,)
Test:  img=(929, 224, 224, 3)  lm=(929, 136)   y_hard=(929,)

4-class hard dist (train): [4526, 416, 287, 58]
Soft label sum per sample: mean=1.0000  min=1.0000


In [3]:
# ── Dataset + loader ──

class SoftImageDS(Dataset):
    def __init__(self, images, y_hard, y_soft):
        self.images = images
        self.y_hard = torch.from_numpy(y_hard).long()
        self.y_soft = torch.from_numpy(y_soft).float()
    def __len__(self): return len(self.y_hard)
    def __getitem__(self, i):
        return (torch.from_numpy(self.images[i]).permute(2, 0, 1).contiguous(),
                self.y_hard[i], self.y_soft[i])


class SoftLandmarkDS(Dataset):
    def __init__(self, landmarks, y_hard, y_soft):
        self.lm = torch.from_numpy(landmarks).float()
        self.y_hard = torch.from_numpy(y_hard).long()
        self.y_soft = torch.from_numpy(y_soft).float()
    def __len__(self): return len(self.y_hard)
    def __getitem__(self, i):
        return self.lm[i], self.y_hard[i], self.y_soft[i]


def make_img_loader(imgs, y_h, y_s, shuffle):
    return DataLoader(SoftImageDS(imgs, y_h, y_s), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0, pin_memory=True)

def make_lm_loader(lm, y_h, y_s, shuffle):
    return DataLoader(SoftLandmarkDS(lm, y_h, y_s), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0, pin_memory=True)


# Loaders (pre-built, reused across configs)
tr_img_loader = make_img_loader(X_tr_img, y_tr, y_tr_soft, True)
v_img_loader  = make_img_loader(X_v_img,  y_v,  y_v_soft,  False)
te_img_loader = make_img_loader(X_te_img, y_te, y_te_soft, False)

tr_lm_loader  = make_lm_loader(X_tr_lm, y_tr, y_tr_soft, True)
v_lm_loader   = make_lm_loader(X_v_lm,  y_v,  y_v_soft,  False)
te_lm_loader  = make_lm_loader(X_te_lm, y_te, y_te_soft, False)

print('Loaders ready.')

Loaders ready.


In [4]:
# ── Loss functions + training loop ──

def kl_div_loss(output, y_h, y_s):
    log_probs = F.log_softmax(output, dim=1)
    return F.kl_div(log_probs, y_s, reduction='batchmean')

def soft_ce_loss(output, y_h, y_s):
    log_probs = F.log_softmax(output, dim=1)
    return -(y_s * log_probs).sum(dim=1).mean()

LOSS_FNS = {'KL_div': kl_div_loss, 'soft_CE': soft_ce_loss}


def train_branch(model_cls, is_cnn, loss_fn, lr, save_path,
                 tr_loader, v_loader):
    model = model_cls(num_classes=NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    best_f1, best_ep, stale = 0.0, 0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        tl, tn = 0.0, 0
        for batch in tr_loader:
            x, y_h, y_s = batch
            x = x.to(device); y_h = y_h.to(device); y_s = y_s.to(device)
            out = model(x)
            loss = loss_fn(out, y_h, y_s)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item() * y_h.size(0); tn += y_h.size(0)
        # Val
        model.eval()
        all_h, all_p = [], []
        with torch.no_grad():
            for batch in v_loader:
                x, y_h, _ = batch
                x = x.to(device); y_h = y_h.to(device)
                all_h.append(y_h.cpu().numpy())
                all_p.append(model(x).argmax(1).cpu().numpy())
        y_h_all = np.concatenate(all_h); y_p_all = np.concatenate(all_p)
        val_f1 = f1_score(y_h_all, y_p_all, average='macro', zero_division=0)
        scheduler.step(val_f1)
        improved = val_f1 > best_f1
        if improved:
            best_f1, best_ep, stale = val_f1, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            stale += 1
        if epoch % 5 == 0 or improved:
            print(f'    ep {epoch:2d}  train_loss={tl/tn:.4f}  val_macroF1={val_f1:.4f} {"*" if improved else ""}')
        if stale >= PATIENCE:
            print(f'    early stop ep {epoch} (best@{best_ep}={best_f1:.4f})')
            break
    return best_f1, best_ep


@torch.no_grad()
def batched_softmax_from_loader(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        x = batch[0].to(device)
        probs.append(torch.softmax(model(x), dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


print('Training helpers ready.')

Training helpers ready.


## Run 2 Configs × 2 Branches

In [5]:
results = {}

for cfg_name, loss_fn in LOSS_FNS.items():
    print(f"\n{'='*70}\n  Config: {cfg_name}\n{'='*70}")
    save_dir = OUTPUT_DIR / f'{NUM_CLASSES}c' / f'LF_TL_{cfg_name}'
    save_dir.mkdir(parents=True, exist_ok=True)
    cnn_path = save_dir / 'cnn_tl.pth'
    fcnn_path = save_dir / 'fcnn.pth'

    # Train CNN TL branch
    print(f'\n  [CNN TL] training with {cfg_name} loss...')
    cnn_best, cnn_ep = train_branch(EmotionCNNTransfer, True, loss_fn, LR_TL, str(cnn_path),
                                     tr_img_loader, v_img_loader)

    # Train FCNN branch
    print(f'\n  [FCNN] training with {cfg_name} loss...')
    fcnn_best, fcnn_ep = train_branch(EmotionFCNN, False, loss_fn, LR_FCNN, str(fcnn_path),
                                       tr_lm_loader, v_lm_loader)

    # Load best checkpoints
    cnn = EmotionCNNTransfer(num_classes=NUM_CLASSES).to(device)
    cnn.load_state_dict(torch.load(cnn_path, map_location=device, weights_only=True))
    fcnn = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
    fcnn.load_state_dict(torch.load(fcnn_path, map_location=device, weights_only=True))

    # Inference on val + test
    v_cnn = batched_softmax_from_loader(cnn, v_img_loader)
    v_fcnn = batched_softmax_from_loader(fcnn, v_lm_loader)
    t_cnn = batched_softmax_from_loader(cnn, te_img_loader)
    t_fcnn = batched_softmax_from_loader(fcnn, te_lm_loader)

    # Grid search w on val
    best_w, best_val_f1 = 0.5, 0.0
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * v_cnn + (1 - w) * v_fcnn).argmax(axis=1)
        f = f1_score(y_v, pr, average='macro', zero_division=0)
        if f > best_val_f1:
            best_val_f1, best_w = f, w
    print(f'\n  Grid search: best w(CNN_TL)={best_w:.2f}  val_macroF1={best_val_f1:.4f}')

    # Evaluate on test
    preds = (best_w * t_cnn + (1 - best_w) * t_fcnn).argmax(axis=1)
    res = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'macro_f1': float(f1_score(y_te, preds, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_te, preds, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_te, preds, average='weighted', zero_division=0)),
        'best_cnn_tl_weight': float(best_w),
        'val_macro_f1': float(best_val_f1),
        'cnn_branch_val_macro_f1': float(cnn_best),
        'fcnn_branch_val_macro_f1': float(fcnn_best),
        'cnn_best_epoch': int(cnn_ep),
        'fcnn_best_epoch': int(fcnn_ep),
    }
    results[f'LateFusionTL_{cfg_name}'] = res
    print(f"  → Test: Macro={res['macro_f1']:.4f}  Micro={res['micro_f1']:.4f}  Weighted={res['weighted_f1']:.4f}  Acc={res['accuracy']:.4f}")
    print(f"  → Per-class F1:")
    print(classification_report(y_te, preds, target_names=EMOTIONS_4, digits=3, zero_division=0))

# Save combined
out_json = OUTPUT_DIR / f'soft_lf_tl_{NUM_CLASSES}c_results.json'
with open(out_json, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {out_json}')


  Config: KL_div

  [CNN TL] training with KL_div loss...


    ep  1  train_loss=0.8610  val_macroF1=0.3835 *


    ep  2  train_loss=0.4040  val_macroF1=0.3943 *


    ep  5  train_loss=0.1498  val_macroF1=0.3602 


    ep 10  train_loss=0.0588  val_macroF1=0.3011 


...

    ep 25  train_loss=0.4287  val_macroF1=0.4410 


    ep 30  train_loss=0.4220  val_macroF1=0.3550 


    ep 35  train_loss=0.4158  val_macroF1=0.3537 


    early stop ep 36 (best@21=0.4709)



  Grid search: best w(CNN_TL)=0.05  val_macroF1=0.4770
  → Test: Macro=0.4323  Micro=0.7987  Weighted=0.7970  Acc=0.7987
  → Per-class F1:
              precision    recall  f1-score   support

     neutral      0.902     0.846     0.873       688
       happy      0.642     0.842     0.728       183
         sad      0.136     0.120     0.128        50
    negative      0.000     0.000     0.000         8

    accuracy                          0.799       929
   macro avg      0.420     0.452     0.432       929
weighted avg      0.802     0.799     0.797       929


Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/soft_label/soft_lf_tl_4c_results.json


## Comparison vs Existing Baselines

In [6]:
print(f"\n{'='*86}")
print(f'  Soft Label Late Fusion TL vs Hard Baselines (4-class, Primer conf60 test)')
print(f"{'='*86}")
print(f"  {'Config':<32} {'Macro':>8} {'Micro':>8} {'Weighted':>10} {'Acc':>8}")
print(f"  {'-'*74}")

# Reference baselines (from existing)
baselines = [
    ('Late Fusion TL B1 hard (existing)', 0.513, 0.802, 0.812, 0.802),
    ('Late Fusion TL B3 hard (target ★)',  0.567, 0.812, 0.821, 0.812),
    ('CNN TL 4c B1 soft KL (nb 71)',       0.517, 0.821, 0.826, 0.821),
]
for name, m, mic, w, a in baselines:
    print(f"  {name:<32} {m:>8.4f} {mic:>8.4f} {w:>10.4f} {a:>8.4f}")

print(f"  {'-'*74}")
for cfg, r in sorted(results.items(), key=lambda kv: -kv[1]['macro_f1']):
    marker = ' ★ beat target' if r['macro_f1'] > 0.567 else ''
    print(f"  {cfg:<32} {r['macro_f1']:>8.4f} {r['micro_f1']:>8.4f} "
          f"{r['weighted_f1']:>10.4f} {r['accuracy']:>8.4f}{marker}")

print(f"\nBranch details:")
for cfg, r in results.items():
    print(f"  {cfg}:")
    print(f"    CNN_TL branch val Macro F1: {r['cnn_branch_val_macro_f1']:.4f} (best ep {r['cnn_best_epoch']})")
    print(f"    FCNN branch   val Macro F1: {r['fcnn_branch_val_macro_f1']:.4f} (best ep {r['fcnn_best_epoch']})")
    print(f"    Best w(CNN_TL): {r['best_cnn_tl_weight']:.2f}  val combined F1: {r['val_macro_f1']:.4f}")


  Soft Label Late Fusion TL vs Hard Baselines (4-class, Primer conf60 test)
  Config                              Macro    Micro   Weighted      Acc
  --------------------------------------------------------------------------
  Late Fusion TL B1 hard (existing)   0.5130   0.8020     0.8120   0.8020
  Late Fusion TL B3 hard (target ★)   0.5670   0.8120     0.8210   0.8120
  CNN TL 4c B1 soft KL (nb 71)       0.5170   0.8210     0.8260   0.8210
  --------------------------------------------------------------------------
  LateFusionTL_KL_div                0.4366   0.8095     0.8007   0.8095
  LateFusionTL_soft_CE               0.4323   0.7987     0.7970   0.7987

Branch details:
  LateFusionTL_KL_div:
    CNN_TL branch val Macro F1: 0.3943 (best ep 2)
    FCNN branch   val Macro F1: 0.4782 (best ep 30)
    Best w(CNN_TL): 0.15  val combined F1: 0.4882
  LateFusionTL_soft_CE:
    CNN_TL branch val Macro F1: 0.4098 (best ep 2)
    FCNN branch   val Macro F1: 0.4709 (best ep 21)
    Best 